# 00 · Data Preparation
Loads raw CSVs → deduplicates → removes metadata leakage → preprocesses (ML & DL variants) → performs a **stratified 70 / 15 / 15 train / val / test split** → saves all artefacts to `outputs/` so every downstream notebook can reload without re-running this one.

In [5]:
import os, re, warnings, random
import numpy as np
import pandas as pd
from tqdm import tqdm
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import nltk
nltk.download('punkt',      quiet=True)
nltk.download('stopwords',  quiet=True)
nltk.download('punkt_tab',  quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

OUTPUT_DIR = '../outputs/preprocessed/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Output dir:', os.path.abspath(OUTPUT_DIR))


Output dir: c:\Users\edward wibowo\UNI\sem4\nlp\aol\ipynb_training\outputs\preprocessed


## 1 · Load & Merge

In [6]:
DATA_DIR = '../dataset/'

fake_df = pd.read_csv(os.path.join(DATA_DIR, 'Fake.csv'))
true_df = pd.read_csv(os.path.join(DATA_DIR, 'True.csv'))

fake_df['label'] = 0   # 0 = Fake
true_df['label'] = 1   # 1 = Real

df = pd.concat([fake_df, true_df], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'Total : {len(df)}  |  Fake : {(df.label==0).sum()}  |  Real : {(df.label==1).sum()}')
df.head(3)


Total : 44898  |  Fake : 23481  |  Real : 21417


,title,text,subject,date,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",0
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",1
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",1


## 2 · Deduplication

In [7]:
print(f'Before dedup : {len(df)}')
n_exact = df.duplicated(subset=['text']).sum()
print(f'Exact duplicates on text : {n_exact}')
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
print(f'After dedup  : {len(df)}')
print(f'Fake : {(df.label==0).sum()}  |  Real : {(df.label==1).sum()}')


Before dedup : 44898
Exact duplicates on text : 6252
After dedup  : 38646
Fake : 17455  |  Real : 21191


## 3 · Remove Metadata / Data-Leakage Tags
Strip publisher headers (e.g. `WASHINGTON (Reuters) -`) **before** any preprocessing to prevent data leakage.

In [8]:
PUBLISHER_PATTERN = re.compile(
    r'^[A-Z\s,\.]+\s*\([^)]+\)\s*[-\u2013]\s*',
    re.MULTILINE
)

def remove_metadata(text: str) -> str:
    if not isinstance(text, str):
        return ''
    return PUBLISHER_PATTERN.sub('', text).strip()

df['text_clean_base'] = df['text'].apply(remove_metadata)

# Combined title + body used by DistilBERT
df['text_combined'] = df['title'].fillna('') + ' ' + df['text_clean_base']

print('Sample after metadata removal:')
print(df['text_clean_base'].iloc[0][:300])


Sample after metadata removal:
21st Century Wire says Ben Stein, reputable professor from, Pepperdine University (also of some Hollywood fame appearing in TV shows and films such as Ferris Bueller s Day Off) made some provocative statements on Judge Jeanine Pirro s show recently. While discussing the halt that was imposed on Pres


## 4 · Preprocessing

| Pipeline | Models | Steps |
|---|---|---|
| `preprocess_ml` | Naive Bayes, SVM | lowercase → URL/IP strip → punctuation strip → stopword removal → Porter stemming |
| `preprocess_dl` | Bi-LSTM, DistilBERT | lowercase → URL/IP strip → punctuation strip (no stemming, no stopword removal) |

In [9]:
stemmer    = PorterStemmer()
stop_words = set(stopwords.words('english'))
URL_RE     = re.compile(r'http\S+|www\.\S+|\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}')
PUNCT_RE   = re.compile(r'[^a-zA-Z\s]')

def preprocess_base(text: str) -> str:
    text = text.lower()
    text = URL_RE.sub(' ', text)
    text = PUNCT_RE.sub(' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def preprocess_ml(text: str) -> str:
    text   = preprocess_base(text)
    tokens = word_tokenize(text)
    tokens = [stemmer.stem(t) for t in tokens if t not in stop_words]
    return ' '.join(tokens)

def preprocess_dl(text: str) -> str:
    return preprocess_base(text)

tqdm.pandas(desc='ML  preprocessing')
df['text_ml'] = df['text_clean_base'].progress_apply(preprocess_ml)

tqdm.pandas(desc='DL  preprocessing')
df['text_dl'] = df['text_clean_base'].progress_apply(preprocess_dl)

tqdm.pandas(desc='BERT preprocessing')
df['text_distilbert'] = df['text_combined'].progress_apply(preprocess_dl)

print('Preprocessing complete.')


BERT preprocessing: 100%|██████████| 38646/38646 [00:07<00:00, 5059.30it/s]

Preprocessing complete.


## 5 · Stratified 70 / 15 / 15 Train / Val / Test Split

**Why three splits?**
- **Train** — model fitting
- **Val** — hyperparameter tuning & early stopping (Bi-LSTM, DistilBERT loop; NB/SVM grid-search target)
- **Test** — held-out final evaluation, **never touched during training**

In [10]:
from sklearn.model_selection import train_test_split

labels = df['label']

# First cut: 70% train, 30% temp
(X_ml_train, X_ml_temp,
 X_dl_train, X_dl_temp,
 X_bert_train, X_bert_temp,
 y_train, y_temp) = train_test_split(
    df['text_ml'], df['text_dl'], df['text_distilbert'], labels,
    test_size=0.30, random_state=SEED, stratify=labels
)

# Second cut: 50/50 → 15% val, 15% test
(X_ml_val, X_ml_test,
 X_dl_val, X_dl_test,
 X_bert_val, X_bert_test,
 y_val, y_test) = train_test_split(
    X_ml_temp, X_dl_temp, X_bert_temp, y_temp,
    test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f'Train : {len(y_train):>6}  ({len(y_train)/len(df)*100:.1f}%)')
print(f'Val   : {len(y_val):>6}  ({len(y_val)/len(df)*100:.1f}%)')
print(f'Test  : {len(y_test):>6}  ({len(y_test)/len(df)*100:.1f}%)')
print()
for split_name, y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    print(f'{split_name}  Fake={( y==0).sum()}  Real={(y==1).sum()}')


Train :  27052  (70.0%)
Val   :   5797  (15.0%)
Test  :   5797  (15.0%)

Train  Fake=12218  Real=14834
Val  Fake=2618  Real=3179
Test  Fake=2619  Real=3178


## 6 · Save Splits

In [ ]:
import pickle

splits = {
    'X_ml_train'   : X_ml_train,
    'X_ml_val'     : X_ml_val,
    'X_ml_test'    : X_ml_test,
    'X_dl_train'   : X_dl_train,
    'X_dl_val'     : X_dl_val,
    'X_dl_test'    : X_dl_test,
    'X_bert_train' : X_bert_train,
    'X_bert_val'   : X_bert_val,
    'X_bert_test'  : X_bert_test,
    'y_train'      : y_train,
    'y_val'        : y_val,
    'y_test'       : y_test,
}

for name, obj in splits.items():
    path = os.path.join(OUTPUT_DIR, f'{name}.pkl')
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    print(f'  Saved {name:20s} → {path}')

print('\nAll splits saved.')


  Saved X_ml_train           → ../outputs/preprocessed/X_ml_train.pkl
  Saved X_ml_val             → ../outputs/preprocessed/X_ml_val.pkl
  Saved X_ml_test            → ../outputs/preprocessed/X_ml_test.pkl
  Saved X_dl_train           → ../outputs/preprocessed/X_dl_train.pkl
  Saved X_dl_val             → ../outputs/preprocessed/X_dl_val.pkl
  Saved X_dl_test            → ../outputs/preprocessed/X_dl_test.pkl
  Saved X_bert_train         → ../outputs/preprocessed/X_bert_train.pkl
  Saved X_bert_val           → ../outputs/preprocessed/X_bert_val.pkl
  Saved X_bert_test          → ../outputs/preprocessed/X_bert_test.pkl
  Saved y_train              → ../outputs/preprocessed/y_train.pkl
  Saved y_val                → ../outputs/preprocessed/y_val.pkl
  Saved y_test               → ../outputs/preprocessed/y_test.pkl

All splits saved.


: 